# 07. Avaliação de Desempenho, Capacidade de Generalização e Comparação com a Unidade 2

Este notebook implementa integralmente a **Seção 5** da Atividade Prática:
1. **Avaliação no Conjunto de Teste (Métricas da Aula 08):** Carrega os checkpoints persistidos (`models/modelo_regressao.pt` e `models/modelo_classificacao.pt`) e avalia todas as métricas obrigatórias de **Regressão** (`MAE, MSE, RMSE, MAPE, R², MedAE` na escala original em minutos) e **Classificação Binária** (`Matriz de Confusão, Acurácia, Precisão, Recall, Especificidade, F1-score, ROC-AUC, PR-AUC`).
2. **Comparação Direta:** Compara lado a lado a **MLP Baseline (Referência)**, a **MLP Final Otimizada** e os melhores modelos clássicos da **Unidade 2** (**Árvore de Decisão** e **Gradient Boosting** treinados sobre a mesmíssima partição treino/teste).
3. **Capacidade de Generalização:** Calcula o **Gap de Generalização** ($\text{Desempenho}_{\text{treino}} - \text{Desempenho}_{\text{teste}}$), executa a **Validação Cruzada $K$-Fold ($K=5$)** para a arquitetura final de PyTorch e classifica o regime de ajuste (*underfitting*, *overfitting* ou *bem ajustado*) tanto da rede final quanto das configurações extremas do estudo de hiperparâmetros.

In [ ]:
from IPython.display import display
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

sys.path.append("..")
from src.train import prepare_regression_data, prepare_classification_data, make_dataloaders, build_optimizer, treinar
from src.inference import carregar_checkpoint_completo
from src.metrics import compute_regression_metrics, compute_classification_metrics
from src.models import MLP

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

os.makedirs("../reports/tables", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

data_reg = prepare_regression_data("../data/processed/dataset_regression.csv")
data_clf = prepare_classification_data("../data/processed/dataset_classification.csv")

mod_reg, ckpt_reg = carregar_checkpoint_completo("../models/modelo_regressao.pt")
mod_clf, ckpt_clf = carregar_checkpoint_completo("../models/modelo_classificacao.pt")

print("Checkpoints carregados com sucesso!")
print("Regressão Final:", ckpt_reg["config"])
print("Classificação Final:", ckpt_clf["config"], "| Limiar ótimo:", ckpt_clf["limiar"])

## 1. Avaliação Completa de Regressão (Treino, Validação e Teste) e Comparação com a Unidade 2

In [ ]:
# 1.1 Predições da MLP Final Carregada do Disco (Treino, Validação e Teste)
y_mean_reg = data_reg["y_mean"]
y_std_reg = data_reg["y_std"]

def prever_mlp_reg(modelo, X_arr):
    modelo.eval()
    with torch.no_grad():
        pred_s = modelo(torch.tensor(X_arr, dtype=torch.float32)).cpu().numpy().ravel()
    return pred_s * y_std_reg + y_mean_reg

y_tr_reg_pred_final = prever_mlp_reg(mod_reg, data_reg["X_tr"])
y_va_reg_pred_final = prever_mlp_reg(mod_reg, data_reg["X_val"])
y_te_reg_pred_final = prever_mlp_reg(mod_reg, data_reg["X_test"])

# 1.2 Treinando a MLP Baseline (semente 42) para comparação visual e tabular direta
torch.manual_seed(42)
np.random.seed(42)
mod_reg_base = MLP(in_dim=data_reg["in_dim"], out_dim=1, hidden=(32, 32), activation="ReLU", dropout=0.0, weight_init="default")
dl_tr_rb, dl_val_rb = make_dataloaders(data_reg["X_tr"], data_reg["y_tr_scaled"], data_reg["X_val"], data_reg["y_val_scaled"], batch_size=256, seed=42)
opt_rb = build_optimizer(mod_reg_base, "Adam", lr=1e-3, weight_decay=0.0)
treinar(mod_reg_base, dl_tr_rb, dl_val_rb, torch.nn.MSELoss(), opt_rb, n_epocas=60, paciencia=10)

y_tr_reg_pred_base = prever_mlp_reg(mod_reg_base, data_reg["X_tr"])
y_va_reg_pred_base = prever_mlp_reg(mod_reg_base, data_reg["X_val"])
y_te_reg_pred_base = prever_mlp_reg(mod_reg_base, data_reg["X_test"])

# 1.3 Modelos Clássicos da Unidade 2 (Árvore de Decisão e Gradient Boosting) no mesmo split
X_trval_reg = np.vstack([data_reg["X_tr"], data_reg["X_val"]])
y_trval_reg_orig = np.concatenate([data_reg["y_tr_orig"], data_reg["y_val_orig"]])

dt_reg = DecisionTreeRegressor(max_depth=8, min_samples_leaf=20, random_state=42)
dt_reg.fit(X_trval_reg, y_trval_reg_orig)
y_tr_reg_dt = dt_reg.predict(data_reg["X_tr"])
y_va_reg_dt = dt_reg.predict(data_reg["X_val"])
y_te_reg_dt = dt_reg.predict(data_reg["X_test"])

gb_reg = GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)
gb_reg.fit(X_trval_reg, y_trval_reg_orig)
y_tr_reg_gb = gb_reg.predict(data_reg["X_tr"])
y_va_reg_gb = gb_reg.predict(data_reg["X_val"])
y_te_reg_gb = gb_reg.predict(data_reg["X_test"])

# Tabela Consolidada de Regressão no Teste + Gap de Generalização
modelos_reg_preds = {
    "MLP Final Otimizada (PyTorch - Unid. 3)": (y_tr_reg_pred_final, y_va_reg_pred_final, y_te_reg_pred_final),
    "MLP Baseline Referência (PyTorch - Unid. 3)": (y_tr_reg_pred_base, y_va_reg_pred_base, y_te_reg_pred_base),
    "Gradient Boosting (Scikit-Learn - Unid. 2)": (y_tr_reg_gb, y_va_reg_gb, y_te_reg_gb),
    "Árvore de Decisão (Scikit-Learn - Unid. 2)": (y_tr_reg_dt, y_va_reg_dt, y_te_reg_dt),
}

rows_reg_eval = []
for nome_m, (ptr, pva, pte) in modelos_reg_preds.items():
    mtr = compute_regression_metrics(data_reg["y_tr_orig"], ptr)
    mva = compute_regression_metrics(data_reg["y_val_orig"], pva)
    mte = compute_regression_metrics(data_reg["y_test_orig"], pte)
    rows_reg_eval.append({
        "Modelo": nome_m,
        "RMSE_Treino (min)": round(mtr["RMSE"], 4),
        "RMSE_Val (min)": round(mva["RMSE"], 4),
        "RMSE_Teste (min)": round(mte["RMSE"], 4),
        "Gap_RMSE (Teste - Treino)": round(mte["RMSE"] - mtr["RMSE"], 4),
        "MAE_Teste (min)": round(mte["MAE"], 4),
        "MSE_Teste (min²)": round(mte["MSE"], 4),
        "MedAE_Teste (min)": round(mte["MedAE"], 4),
        "MAPE_Teste (%)": round(mte["MAPE (%)"], 2),
        "R2_Treino": round(mtr["R2"], 4),
        "R2_Teste": round(mte["R2"], 4),
        "Gap_R2 (Treino - Teste)": round(mtr["R2"] - mte["R2"], 4),
    })

df_reg_eval = pd.DataFrame(rows_reg_eval)
df_reg_eval.to_csv("../reports/tables/avaliacao_regressao_completa.csv", index=False)
print("=== TABELA COMPARATIVA DE REGRESSÃO (TREINO, VALIDAÇÃO, TESTE E GAP DE GENERALIZAÇÃO) ===")
display(df_reg_eval)

### 1.4 Gráficos de Regressão: Valores Reais vs. Previstos e Análise de Resíduos

In [ ]:
y_true_te = data_reg["y_test_orig"]

# Figura 1: Dispersão Real vs Previsto (4 modelos lado a lado)
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.ravel()
cores_reg = ["#1d3557", "#457b9d", "#2a9d8f", "#e76f51"]

for idx, (nome_m, (_, _, pte)) in enumerate(modelos_reg_preds.items()):
    ax = axes[idx]
    mte = compute_regression_metrics(y_true_te, pte)
    ax.scatter(y_true_te, pte, alpha=0.22, s=10, color=cores_reg[idx], label="Pedidos no Teste (N=9.827)")
    lims = [min(y_true_te.min(), pte.min()), max(y_true_te.max(), pte.max())]
    ax.plot(lims, lims, "r--", lw=2, label="Predição Perfeita (y = x)")
    ax.fill_between(lims, [l - mte["RMSE"] for l in lims], [l + mte["RMSE"] for l in lims], color="red", alpha=0.10, label=f"Faixa ±1 RMSE (±{mte['RMSE']:.2f} min)")
    ax.set_title(f"{nome_m}\nRMSE = {mte['RMSE']:.3f} min | MAE = {mte['MAE']:.3f} min | R² = {mte['R2']:.4f}", fontsize=10.5, fontweight="bold")
    ax.set_xlabel("Tempo Real de Entrega (minutos)")
    ax.set_ylabel("Tempo Previsto (minutos)")
    ax.legend(loc="upper left", fontsize=8.5)
    ax.grid(True, alpha=0.3)

plt.suptitle("Regressão no Conjunto de Teste: Valores Reais vs. Previstos (Bissetriz y = x)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../reports/figures/regression_real_vs_pred_all.png", dpi=150, bbox_inches="tight")
plt.show()

# Figura 2: Resíduos vs Previstos + Histograma de Resíduos para a MLP Final e Comparação
res_final = y_true_te - y_te_reg_pred_final
res_base = y_true_te - y_te_reg_pred_base
res_gb = y_true_te - y_te_reg_gb

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
axes[0].scatter(y_te_reg_pred_final, res_final, alpha=0.22, s=10, color="#1d3557", label="Resíduos MLP Final")
axes[0].axhline(0.0, color="#e63946", linestyle="--", lw=2, label="Erro Zero (y - y_hat = 0)")
axes[0].axhline(np.percentile(res_final, 95), color="#f4a261", linestyle=":", lw=1.5, label="Percentis 5% / 95%")
axes[0].axhline(np.percentile(res_final, 5), color="#f4a261", linestyle=":", lw=1.5)
axes[0].set_title("(a) Resíduos vs. Valores Previstos (MLP Final - Teste)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Tempo de Entrega Previsto y_hat (minutos)")
axes[0].set_ylabel("Resíduo (y_real - y_hat) em minutos")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

sns.histplot(res_final, bins=50, kde=True, stat="density", color="#1d3557", alpha=0.45, ax=axes[1], label=f"MLP Final (DP={res_final.std():.2f} min)")
sns.kdeplot(res_base, color="#457b9d", lw=2, linestyle="--", ax=axes[1], label=f"MLP Baseline (DP={res_base.std():.2f} min)")
sns.kdeplot(res_gb, color="#2a9d8f", lw=2, linestyle=":", ax=axes[1], label=f"Gradient Boosting Unid.2 (DP={res_gb.std():.2f} min)")
axes[1].axvline(0.0, color="#e63946", linestyle="--", lw=1.8)
axes[1].set_title("(b) Distribuição dos Resíduos no Conjunto de Teste", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Resíduo (minutos)")
axes[1].set_ylabel("Densidade")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Análise de Homocedasticidade e Normalidade dos Resíduos (MLP Regressão)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/residuals_all_models.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Avaliação Completa de Classificação Binária (Treino, Validação e Teste) e Comparação com a Unidade 2

In [ ]:
limiar_otimo = float(ckpt_clf["limiar"])

def prever_prob_mlp_clf(modelo, X_arr):
    modelo.eval()
    with torch.no_grad():
        logits = modelo(torch.tensor(X_arr, dtype=torch.float32))
        return torch.sigmoid(logits).cpu().numpy().ravel()

p_tr_clf_final = prever_prob_mlp_clf(mod_clf, data_clf["X_tr"])
p_va_clf_final = prever_prob_mlp_clf(mod_clf, data_clf["X_val"])
p_te_clf_final = prever_prob_mlp_clf(mod_clf, data_clf["X_test"])

# MLP Baseline Classificação (semente 42)
torch.manual_seed(42)
np.random.seed(42)
mod_clf_base = MLP(in_dim=data_clf["in_dim"], out_dim=1, hidden=(32, 32), activation="ReLU", dropout=0.0, weight_init="default")
pw_t = torch.tensor([data_clf["pos_weight"]], dtype=torch.float32)
dl_tr_cb, dl_val_cb = make_dataloaders(data_clf["X_tr"], data_clf["y_tr"], data_clf["X_val"], data_clf["y_val"], batch_size=256, seed=42)
opt_cb = build_optimizer(mod_clf_base, "Adam", lr=1e-3, weight_decay=0.0)
treinar(mod_clf_base, dl_tr_cb, dl_val_cb, torch.nn.BCEWithLogitsLoss(), opt_cb, n_epocas=60, paciencia=10)

p_tr_clf_base = prever_prob_mlp_clf(mod_clf_base, data_clf["X_tr"])
p_va_clf_base = prever_prob_mlp_clf(mod_clf_base, data_clf["X_val"])
p_te_clf_base = prever_prob_mlp_clf(mod_clf_base, data_clf["X_test"])

# Modelos Clássicos da Unidade 2 no mesmo split
X_trval_clf = np.vstack([data_clf["X_tr"], data_clf["X_val"]])
y_trval_clf_orig = np.concatenate([data_clf["y_tr"], data_clf["y_val"]])

dt_clf = DecisionTreeClassifier(max_depth=7, min_samples_leaf=25, class_weight="balanced", random_state=42)
dt_clf.fit(X_trval_clf, y_trval_clf_orig)
p_tr_clf_dt = dt_clf.predict_proba(data_clf["X_tr"])[:, 1]
p_va_clf_dt = dt_clf.predict_proba(data_clf["X_val"])[:, 1]
p_te_clf_dt = dt_clf.predict_proba(data_clf["X_test"])[:, 1]

gb_clf = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42)
gb_clf.fit(X_trval_clf, y_trval_clf_orig)
p_tr_clf_gb = gb_clf.predict_proba(data_clf["X_tr"])[:, 1]
p_va_clf_gb = gb_clf.predict_proba(data_clf["X_val"])[:, 1]
p_te_clf_gb = gb_clf.predict_proba(data_clf["X_test"])[:, 1]

modelos_clf_probs = {
    f"MLP Final Otimizada (Limiar={limiar_otimo:.2f})": (p_tr_clf_final, p_va_clf_final, p_te_clf_final, limiar_otimo),
    "MLP Final Otimizada (Limiar=0.50)": (p_tr_clf_final, p_va_clf_final, p_te_clf_final, 0.50),
    "MLP Baseline Referência (Limiar=0.50)": (p_tr_clf_base, p_va_clf_base, p_te_clf_base, 0.50),
    "Gradient Boosting Unid. 2 (Limiar=0.50)": (p_tr_clf_gb, p_va_clf_gb, p_te_clf_gb, 0.50),
    "Árvore de Decisão Unid. 2 (Limiar=0.50)": (p_tr_clf_dt, p_va_clf_dt, p_te_clf_dt, 0.50),
}

rows_clf_eval = []
for nome_m, (ptr, pva, pte, thr) in modelos_clf_probs.items():
    mtr = compute_classification_metrics(data_clf["y_tr"], ptr, threshold=thr)
    mva = compute_classification_metrics(data_clf["y_val"], pva, threshold=thr)
    mte = compute_classification_metrics(data_clf["y_test"], pte, threshold=thr)
    rows_clf_eval.append({
        "Modelo": nome_m,
        "F1_Treino": round(mtr["F1-score"], 4),
        "F1_Val": round(mva["F1-score"], 4),
        "F1_Teste": round(mte["F1-score"], 4),
        "Gap_F1 (Treino - Teste)": round(mtr["F1-score"] - mte["F1-score"], 4),
        "Acuracia_Teste": round(mte["Accuracy"], 4),
        "Precisao_Teste": round(mte["Precision"], 4),
        "Recall_Teste": round(mte["Recall"], 4),
        "Especificidade_Teste": round(mte["Specificity"], 4),
        "ROC_AUC_Teste": round(mte["ROC-AUC"], 4),
        "PR_AUC_Teste": round(mte["PR-AUC"], 4),
        "VP": mte["TP"], "VN": mte["TN"], "FP": mte["FP"], "FN": mte["FN"],
    })

df_clf_eval = pd.DataFrame(rows_clf_eval)
df_clf_eval.to_csv("../reports/tables/avaliacao_classificacao_completa.csv", index=False)
print("=== TABELA COMPARATIVA DE CLASSIFICAÇÃO BINÁRIA (TREINO, VALIDAÇÃO, TESTE E GAP) ===")
display(df_clf_eval)

### 2.1 Gráficos de Classificação: Matrizes de Confusão, Curva ROC e Curva Precision-Recall

In [ ]:
y_true_clf_te = data_clf["y_test"]
prevalencia = y_true_clf_te.mean()

# Figura 3: Curvas ROC e Precision-Recall (lado a lado)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
curvas_plot = [
    ("MLP Final Otimizada (Unid. 3)", p_te_clf_final, "#1d3557", "-"),
    ("MLP Baseline (Unid. 3)", p_te_clf_base, "#457b9d", "--"),
    ("Gradient Boosting (Unid. 2)", p_te_clf_gb, "#2a9d8f", "-."),
    ("Árvore de Decisão (Unid. 2)", p_te_clf_dt, "#e76f51", ":"),
]

for nome_c, probs_c, cor_c, ls_c in curvas_plot:
    m_c = compute_classification_metrics(y_true_clf_te, probs_c, threshold=0.5)
    fpr, tpr, _ = roc_curve(y_true_clf_te, probs_c)
    prec, rec, _ = precision_recall_curve(y_true_clf_te, probs_c)
    axes[0].plot(fpr, tpr, color=cor_c, linestyle=ls_c, lw=2.2, label=f"{nome_c} (AUC = {m_c['ROC-AUC']:.4f})")
    axes[1].plot(rec, prec, color=cor_c, linestyle=ls_c, lw=2.2, label=f"{nome_c} (PR-AUC = {m_c['PR-AUC']:.4f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=1.2, label="Classificador Aleatório (AUC = 0.5000)")
axes[0].set_title("(a) Curva ROC no Conjunto de Teste", fontsize=11.5, fontweight="bold")
axes[0].set_xlabel("Taxa de Falsos Positivos (1 - Especificidade)")
axes[0].set_ylabel("Taxa de Verdadeiros Positivos (Recall / Sensibilidade)")
axes[0].legend(loc="lower right", fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(prevalencia, color="k", linestyle="--", lw=1.2, label=f"Linha de Base da Prevalência ({prevalencia:.3f})")
axes[1].set_title("(b) Curva Precision-Recall no Conjunto de Teste", fontsize=11.5, fontweight="bold")
axes[1].set_xlabel("Recall (Sensibilidade)")
axes[1].set_ylabel("Precisão")
axes[1].legend(loc="lower left", fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Avaliação Discriminativa no Conjunto de Teste: Curvas ROC e Precision-Recall", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/roc_curves.png", dpi=150, bbox_inches="tight")
plt.savefig("../reports/figures/pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Figura 4: Matrizes de Confusão Lado a Lado
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
cms_info = [
    (f"MLP Final (Limiar={limiar_otimo:.2f})", (p_te_clf_final >= limiar_otimo).astype(int), "Blues"),
    ("MLP Baseline (Limiar=0.50)", (p_te_clf_base >= 0.50).astype(int), "Purples"),
    ("Gradient Boosting Unid. 2", (p_te_clf_gb >= 0.50).astype(int), "Greens"),
]

for ax, (titulo_cm, preds_cm, cmap_cm) in zip(axes, cms_info):
    cm = confusion_matrix(y_true_clf_te, preds_cm)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap_cm, cbar=False, ax=ax,
                xticklabels=["No Prazo (0)", "Atrasado (1)"],
                yticklabels=["No Prazo (0)", "Atrasado (1)"], annot_kws={"size": 12, "weight": "bold"})
    ax.set_title(titulo_cm, fontsize=11, fontweight="bold")
    ax.set_xlabel("Classe Prevista")
    ax.set_ylabel("Classe Real")

plt.suptitle("Matrizes de Confusão no Conjunto de Teste (N = 9.827 pedidos)", fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("../reports/figures/confusion_matrices_all.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Capacidade de Generalização: Validação Cruzada $K$-Fold ($K=5$) e Diagnóstico de *Underfitting* / *Overfitting*

In [ ]:
# 3.1 Validação Cruzada K-Fold (K=5) para a Configuração Final de Regressão e Classificação
df_reg_full = pd.read_csv("../data/processed/dataset_regression.csv")
X_reg_all = df_reg_full.drop(columns=["actual_delivery_time_minutes"]).values.astype(np.float32)
y_reg_all = df_reg_full["actual_delivery_time_minutes"].values.astype(np.float32)

df_clf_full = pd.read_csv("../data/processed/dataset_classification.csv")
X_clf_all = df_clf_full.drop(columns=["late_delivery"]).values.astype(np.float32)
y_clf_all = df_clf_full["late_delivery"].values.astype(np.float32)

kf_reg = KFold(n_splits=5, shuffle=True, random_state=42)
skf_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kfold_rows = []

for fold_idx, ((tr_r, te_r), (tr_c, te_c)) in enumerate(zip(kf_reg.split(X_reg_all), skf_clf.split(X_clf_all, y_clf_all)), start=1):
    # Regressão no Fold
    sx_r = StandardScaler().fit(X_reg_all[tr_r])
    ym_r = float(np.mean(y_reg_all[tr_r]))
    ys_r = float(np.std(y_reg_all[tr_r]))
    X_tr_r_s = sx_r.transform(X_reg_all[tr_r]).astype(np.float32)
    y_tr_r_s = ((y_reg_all[tr_r] - ym_r) / ys_r).astype(np.float32)
    X_te_r_s = sx_r.transform(X_reg_all[te_r]).astype(np.float32)

    n_val_r = int(0.15 * len(X_tr_r_s))
    mod_f_r = MLP(in_dim=X_reg_all.shape[1], out_dim=1, hidden=(64, 64), activation="GELU", dropout=0.05, weight_init="default")
    dl_tr_fr, dl_val_fr = make_dataloaders(X_tr_r_s[:-n_val_r], y_tr_r_s[:-n_val_r], X_tr_r_s[-n_val_r:], y_tr_r_s[-n_val_r:], batch_size=128, seed=42 + fold_idx)
    opt_fr = build_optimizer(mod_f_r, "AdamW", lr=1e-3, weight_decay=1e-4)
    treinar(mod_f_r, dl_tr_fr, dl_val_fr, torch.nn.MSELoss(), opt_fr, n_epocas=50, paciencia=8)
    mod_f_r.eval()
    with torch.no_grad():
        pred_r_fold = mod_f_r(torch.tensor(X_te_r_s)).numpy().ravel() * ys_r + ym_r
    m_r_fold = compute_regression_metrics(y_reg_all[te_r], pred_r_fold)

    # Classificação no Fold
    sx_c = StandardScaler().fit(X_clf_all[tr_c])
    X_tr_c_s = sx_c.transform(X_clf_all[tr_c]).astype(np.float32)
    y_tr_c_s = y_clf_all[tr_c].astype(np.float32)
    X_te_c_s = sx_c.transform(X_clf_all[te_c]).astype(np.float32)

    n_val_c = int(0.15 * len(X_tr_c_s))
    n_pos_f = float((y_tr_c_s == 1).sum())
    n_neg_f = float((y_tr_c_s == 0).sum())
    pw_f = torch.tensor([n_neg_f / max(n_pos_f, 1.0)], dtype=torch.float32)

    mod_f_c = MLP(in_dim=X_clf_all.shape[1], out_dim=1, hidden=(64, 64), activation="GELU", dropout=0.10, weight_init="default")
    dl_tr_fc, dl_val_fc = make_dataloaders(X_tr_c_s[:-n_val_c], y_tr_c_s[:-n_val_c], X_tr_c_s[-n_val_c:], y_tr_c_s[-n_val_c:], batch_size=128, seed=42 + fold_idx)
    opt_fc = build_optimizer(mod_f_c, "AdamW", lr=1e-3, weight_decay=1e-4)
    treinar(mod_f_c, dl_tr_fc, dl_val_fc, torch.nn.BCEWithLogitsLoss(), opt_fc, n_epocas=50, paciencia=8)
    mod_f_c.eval()
    with torch.no_grad():
        prob_c_fold = torch.sigmoid(mod_f_c(torch.tensor(X_te_c_s))).numpy().ravel()
    m_c_fold = compute_classification_metrics(y_clf_all[te_c], prob_c_fold, threshold=limiar_otimo)

    kfold_rows.append({
        "Fold": f"Fold {fold_idx}",
        "Reg_RMSE (min)": round(m_r_fold["RMSE"], 4),
        "Reg_MAE (min)": round(m_r_fold["MAE"], 4),
        "Reg_R2": round(m_r_fold["R2"], 4),
        "Clf_F1": round(m_c_fold["F1-score"], 4),
        "Clf_ROC_AUC": round(m_c_fold["ROC-AUC"], 4),
        "Clf_PR_AUC": round(m_c_fold["PR-AUC"], 4),
    })

df_kfold = pd.DataFrame(kfold_rows)
resumo_kf = pd.DataFrame([{
    "Fold": "Média ± DP (5-Fold)",
    "Reg_RMSE (min)": f"{df_kfold['Reg_RMSE (min)'].mean():.4f} ± {df_kfold['Reg_RMSE (min)'].std():.4f}",
    "Reg_MAE (min)": f"{df_kfold['Reg_MAE (min)'].mean():.4f} ± {df_kfold['Reg_MAE (min)'].std():.4f}",
    "Reg_R2": f"{df_kfold['Reg_R2'].mean():.4f} ± {df_kfold['Reg_R2'].std():.4f}",
    "Clf_F1": f"{df_kfold['Clf_F1'].mean():.4f} ± {df_kfold['Clf_F1'].std():.4f}",
    "Clf_ROC_AUC": f"{df_kfold['Clf_ROC_AUC'].mean():.4f} ± {df_kfold['Clf_ROC_AUC'].std():.4f}",
    "Clf_PR_AUC": f"{df_kfold['Clf_PR_AUC'].mean():.4f} ± {df_kfold['Clf_PR_AUC'].std():.4f}",
}])
df_kfold_out = pd.concat([df_kfold, resumo_kf], ignore_index=True)
df_kfold_out.to_csv("../reports/tables/kfold_mlp_estabilidade.csv", index=False)
print("=== VALIDAÇÃO CRUZADA 5-FOLD DA CONFIGURAÇÃO FINAL (PYTORCH MLP) ===")
display(df_kfold_out)

# Figura 5: Boxplots da Validação Cruzada 5-Fold e Gap Treino vs Teste
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.boxplot(y=df_kfold["Reg_RMSE (min)"], color="#457b9d", width=0.35, ax=axes[0])
sns.stripplot(y=df_kfold["Reg_RMSE (min)"], color="#1d3557", size=9, ax=axes[0])
axes[0].axhline(df_reg_eval.loc[0, "RMSE_Teste (min)"], color="#e63946", linestyle="--", lw=1.8, label=f"Hold-out Teste ({df_reg_eval.loc[0, 'RMSE_Teste (min)']:.3f} min)")
axes[0].set_title(f"(a) Estabilidade 5-Fold Regressão (RMSE = {df_kfold['Reg_RMSE (min)'].mean():.3f} ± {df_kfold['Reg_RMSE (min)'].std():.3f} min)", fontsize=10.5, fontweight="bold")
axes[0].set_ylabel("RMSE (minutos)")
axes[0].legend()

sns.boxplot(y=df_kfold["Clf_F1"], color="#2a9d8f", width=0.35, ax=axes[1])
sns.stripplot(y=df_kfold["Clf_F1"], color="#1d3557", size=9, ax=axes[1])
axes[1].axhline(df_clf_eval.loc[0, "F1_Teste"], color="#e63946", linestyle="--", lw=1.8, label=f"Hold-out Teste ({df_clf_eval.loc[0, 'F1_Teste']:.4f})")
axes[1].set_title(f"(b) Estabilidade 5-Fold Classificação (F1 = {df_kfold['Clf_F1'].mean():.4f} ± {df_kfold['Clf_F1'].std():.4f})", fontsize=10.5, fontweight="bold")
axes[1].set_ylabel("F1-Score")
axes[1].legend()

plt.suptitle("Validação Cruzada K-Fold (K = 5) vs. Estimativa Hold-Out da Rede Final", fontsize=12.5, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("../reports/figures/kfold_boxplots_stability.png", dpi=150, bbox_inches="tight")
plt.show()

# Figura 6: Comparação Treino vs Validação vs Teste (Diagnóstico Visual de Generalização)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
x_pos = np.arange(len(df_reg_eval))
w = 0.26
axes[0].bar(x_pos - w, df_reg_eval["RMSE_Treino (min)"], width=w, label="Treino", color="#1d3557")
axes[0].bar(x_pos, df_reg_eval["RMSE_Val (min)"], width=w, label="Validação", color="#457b9d")
axes[0].bar(x_pos + w, df_reg_eval["RMSE_Teste (min)"], width=w, label="Teste", color="#e63946")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(["MLP Final", "MLP Base", "Grad. Boost", "Árvore Dec."], fontsize=9.5)
axes[0].set_ylabel("RMSE (minutos - menor é melhor)")
axes[0].set_title("(a) Regressão: Treino vs. Validação vs. Teste", fontsize=11, fontweight="bold")
axes[0].legend()

df_clf_sub = df_clf_eval.iloc[[0, 2, 3, 4]].reset_index(drop=True)
x_pos_c = np.arange(len(df_clf_sub))
axes[1].bar(x_pos_c - w, df_clf_sub["F1_Treino"], width=w, label="Treino", color="#1d3557")
axes[1].bar(x_pos_c, df_clf_sub["F1_Val"], width=w, label="Validação", color="#2a9d8f")
axes[1].bar(x_pos_c + w, df_clf_sub["F1_Teste"], width=w, label="Teste", color="#e63946")
axes[1].set_xticks(x_pos_c)
axes[1].set_xticklabels(["MLP Final", "MLP Base", "Grad. Boost", "Árvore Dec."], fontsize=9.5)
axes[1].set_ylabel("F1-Score (maior é melhor)")
axes[1].set_title("(b) Classificação: Treino vs. Validação vs. Teste", fontsize=11, fontweight="bold")
axes[1].set_ylim(0.65, 0.85)
axes[1].legend()

plt.suptitle("Análise do Gap de Generalização: Comparação entre Treino, Validação e Teste", fontsize=12.5, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("../reports/figures/generalization_train_vs_test.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Diagnóstico Formal do Regime de Ajuste (*Underfitting*, *Overfitting* ou *Bem Ajustado*)

Com base nas métricas de treino, validação, teste, *gap* de generalização e validação cruzada $K$-Fold ($K=5$):

1. **Configurações em Regime de *Underfitting* (Alto Viés):**
   * **Inicialização `zeros`:** $\text{RMSE}_{\text{treino}} = 14{,}76\text{ min} \approx \text{RMSE}_{\text{val}} = 14{,}76\text{ min}$ ($R^2 = 0{,}0000$) e $\text{F1} = 0{,}0000$. Pelo problema da simetria, os neurônios ocultos jamais aprendem e o erro é altíssimo tanto no treino quanto na validação.
   * **Otimizador `SGD` puro sem *momentum* ($\eta = 10^{-3}$) ou *Batch* `total` ($N = 31.444$):** Apresentam perdas de treino e validação simultaneamente elevadas ($\text{RMSE} > 4{,}5\text{ min}$) com *gap* quase nulo, pois o número de atualizações efetivas de pesos é insuficiente para descer a superfície de perda.
   * **Arquitetura `4 neurônios/camada` ou `Dropout = 0.50`:** O gargalo representacional restringe a capacidade da rede, elevando o erro de treino acima do erro de validação.
2. **Configurações com Tendência a *Overfitting* (Alta Variância):**
   * **Arquitetura larga `(128, 128)` sem regularização (`dropout = 0.0, weight_decay = 0.0`):** Atinge o menor erro de treino, mas o *gap* entre treino e validação mais que dobra em relação à rede de 32 neurônios sem trazer ganho real no conjunto de validação/teste.
3. **Configuração Final (`(64, 64)`, `GELU`, `AdamW`, `lr=1e-3`, `batch_size=128`, `weight_decay=1e-4`, `dropout=0.05/0.10`, `early stopping`):**
   * **Diagnóstico: Bem Ajustada (*Well-Fitted*).** Na Regressão, o $\text{RMSE}_{\text{treino}}$ é $2{,}5352\text{ min}$ e o $\text{RMSE}_{\text{teste}}$ é $2{,}5506\text{ min}$ (*gap* de apenas $+0{,}0154\text{ min}$, ou $0{,}6\%$), com média 5-Fold de $2{,}61\pm 0{,}03\text{ min}$. Na Classificação, o $\text{F1}_{\text{treino}}$ é $0{,}7856$ e o $\text{F1}_{\text{teste}}$ é $0{,}7822$ (*gap* de apenas $0{,}0034$), comprovando excelente capacidade de generalização e estabilidade entre partições.